In [1]:
from singleCAM_IROS._pipeline_support import _handle_dirpaths
from typing import Callable

from pathlib import Path

from numpy.typing import NDArray
import numpy as np
from astropy.io.fits.fitsrec import FITS_rec

from bloodmoon.io import simulation_files
from bloodmoon.mask import CodedMaskCamera, codedmask, count
from bloodmoon.images import argmax

import darksun as ds

ds.show.set_figures_darkbkg()

In [2]:
#MASK_FITS: str = "mask_050_1040x17_20260129_CORRECTED.fits"
MASK_FITS: str = "wfm_mask_NTHT_20250725.fits"

SKYFIELD: str = "IROSDummy"
#SKYFIELD: str = "LMC"
#DATA_FITS: str = "crab_mask_050_1040x17_20260129_2-50keV_1ks"
DATA_FITS: str = "iros_benchmark_2-50keV_mask_050_1040x17_infdet_1ks"
#DATA_FITS: str = "lmc_rxte-sax_mask_050_1040x17_2-50keV_1ks"

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"

UPS_X: int = 10
UPS_Y: int = 5

In [3]:
# load filepaths
mask_path, simul_data, save_path = _handle_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
)
wfm: CodedMaskCamera = codedmask(mask_path, UPS_X, UPS_Y)
filepaths: dict[str, dict[str, Path]] = simulation_files(simul_data)


# data from camera A
catalogueA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlA_detct = ds.get_data(filepaths[ID_CAMERA_A]['detected'])
sdlA_recnstr = ds.get_data(filepaths[ID_CAMERA_A]['reconstructed'])

## data from camera B
catalogueB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])
sdlB_detct = ds.get_data(filepaths[ID_CAMERA_B]['detected'])
sdlB_recnstr = ds.get_data(filepaths[ID_CAMERA_B]['reconstructed'])

In [4]:
def get_phs_IDs(data: FITS_rec) -> NDArray:
    """Extracts the photons IDs and status flags from given phs record."""
    return data['ID']


detected_IDs_camA: NDArray = get_phs_IDs(sdlA_detct.DLdata)
reconsrt_IDs_camA: NDArray = get_phs_IDs(sdlA_recnstr.DLdata)

detected_IDs_camB: NDArray = get_phs_IDs(sdlB_detct.DLdata)
reconsrt_IDs_camB: NDArray = get_phs_IDs(sdlB_recnstr.DLdata)


unqIDs_dtc_camA, unqIDs_rct_camA = map(lambda x: len(np.unique(x)), (detected_IDs_camA, reconsrt_IDs_camA))
unqIDs_dtc_camB, unqIDs_rct_camB = map(lambda x: len(np.unique(x)), (detected_IDs_camB, reconsrt_IDs_camB))

print(
    f'## Camera: {ID_CAMERA_A.upper()}\n'
    f'  * IDs in detected list: {len(detected_IDs_camA)}\n'
    f'  * Unique IDs in detected list: {unqIDs_dtc_camA}\n'
    f'  * Delta (detc - unique detc): {len(detected_IDs_camA) - unqIDs_dtc_camA}\n\n'

    f'  * Unique IDs in reconstr list: {unqIDs_rct_camA}\n'
    f'  * Delta (detc - rectr): {unqIDs_dtc_camA - unqIDs_rct_camA}\n\n'


    f'## Camera: {ID_CAMERA_B.upper()}\n'
    f'  * IDs in detected list: {len(detected_IDs_camB)}\n'
    f'  * Unique IDs in detected list: {unqIDs_dtc_camB}\n'
    f'  * Delta (detc - unique detc): {len(detected_IDs_camB) - unqIDs_dtc_camB}\n\n'

    f'  * Unique IDs in reconstr list: {unqIDs_rct_camB}\n'
    f'  * Delta (detc - rectr): {unqIDs_dtc_camB - unqIDs_rct_camB}\n\n'
)

## Camera: CAM1A
  * IDs in detected list: 4088323
  * Unique IDs in detected list: 3688399
  * Delta (detc - unique detc): 399924

  * Unique IDs in reconstr list: 3682975
  * Delta (detc - rectr): 5424

## Camera: CAM1B
  * IDs in detected list: 4404313
  * Unique IDs in detected list: 4267360
  * Delta (detc - unique detc): 136953

  * Unique IDs in reconstr list: 4259557
  * Delta (detc - rectr): 7803




In [ ]:
from typing import NamedTuple

class Duplicate(NamedTuple):
    """
    Identifier for duplicated entries in record.

    Attributes:
        value (int): Entry value.
        rpts (int): Number of entry repetitions.
        idxs (slice): Value idxs in array.
    """
    value: int
    rpts: int
    idxs: slice


def organise_phs_list(record: FITS_rec) -> FITS_rec:
    """Sorts the given photons list by ID."""
    return np.sort(record, order='ID')

def check_phs_list_degeneracy(record: FITS_rec, verbose: bool = True) -> tuple[Duplicate, ...]:
    """Checks the given **ID sorted** list of photons for repeating entries."""
    unq, idxs, cts = np.unique(record['ID'], return_index=True, return_counts=True)
    rpts_vals = np.where((cts > 1))[0]
    if verbose:
        print(
            f'Unique record entries: {len(unq)} / {len(record)} ({len(unq) * 100 / len(record):.2f} %)\n'
            f'Repeating record entries: {len(rpts_vals)}\n'
        )
    duplicates = tuple(
        Duplicate(unq[rpts], cts[rpts], slice(idxs[rpts], idxs[rpts] + cts[rpts]))
        for rpts in rpts_vals
    )
    return duplicates

In [23]:
sorted_phs_list: FITS_rec = organise_phs_list(sdlA_detct.DLdata)
duplicates: tuple[Duplicate, ...] = check_phs_list_degeneracy(sorted_phs_list)

Unique record entries: 3688399 / 4088323 (90.21789618873069 %)
Repeating record entries: 274563



In [25]:
idx: int = 0

sorted_phs_list[duplicates[idx].idxs]

FITS_rec([(464, -57.02303193, -56.70231963, 0.2249312 , 0.28843557, -0.23682338, -0.92774975, 9.16388712, 800.50827171, 246.99546814, -44.53539658),
          (464, -57.02301266, -69.61680178, 0.22486923, 0.28843557, -0.23682338, -0.92774975, 9.16388712, 800.50827171, 246.99546814, -44.53539658)],
         dtype=(numpy.record, [('ID', '>i8'), ('X', '>f8'), ('Y', '>f8'), ('Z', '>f8'), ('DIRX', '>f8'), ('DIRY', '>f8'), ('DIRZ', '>f8'), ('ENERGY', '>f8'), ('TIME', '>f8'), ('RA', '>f8'), ('DEC', '>f8')]))